# Загрузка данных

In [1]:
from corus import load_factru
import re
from pathlib import Path

ROOT_DIR = Path.cwd() 
DATA_DIR = ROOT_DIR / "data" / "factRuEval-2016-master"

records = list(load_factru(str(DATA_DIR)))
print(f"Загружено записей: {len(records)}")

def whitespace_tokenize_with_offsets(text: str):
    tokens = []
    spans = []
    for m in re.finditer(r'\S+', text):
        tokens.append(m.group())
        spans.append((m.start(), m.end()))
    return tokens, spans

Загружено записей: 254


In [2]:
unique_labels = set()
for record in records:
    for obj in record.objects:
        unique_labels.add(obj.type)
print("Уникальные метки в исходном датасете:\n", unique_labels)

Уникальные метки в исходном датасете:
 {'LocOrg', 'Facility', 'Project', 'Org', 'Person', 'Location'}


# Формирование обучающих данных

In [3]:
# Переведём в PER/ORG/LOC/MISC

# Универсальная функция для маппинга типов объектов к базовым NER-классам
def map_object_type(obj_type: str) -> str:
    t = (obj_type or "").lower()
    if "person" in t or t in {"person", "name", "surname", "firstname", "patronymic"}:
        return "PER"
    if "org" in t or "organization" in t or "company" in t or "org_name" in t or "org_descr" in t:
        return "ORG"
    if "loc" in t or "location" in t or "geo" in t or "place" in t or "loc_name" in t:
        return "LOC"
    return "MISC"

In [4]:
# Переведём в BIO формат

examples = []
for rec in records:
    text = rec.text
    tokens, token_spans = whitespace_tokenize_with_offsets(text)
    token_labels = ["O"] * len(tokens)

    for obj in rec.objects:
        base_type = map_object_type(obj.type)
        for span in obj.spans:
            span_start = span.start
            span_end = span.stop
            overlapping_idxs = []
            for i, (t_start, t_end) in enumerate(token_spans):
                if not (t_end <= span_start or t_start >= span_end):
                    overlapping_idxs.append(i)
            if not overlapping_idxs:
                # можно логировать: print(f"No overlap for span {span_start}-{span_end} in doc {rec.id}")
                continue
            for j, tok_idx in enumerate(overlapping_idxs):
                if token_labels[tok_idx] != "O":
                    continue
                prefix = "B" if j == 0 else "I"
                token_labels[tok_idx] = f"{prefix}-{base_type}"

    examples.append({
        "id": rec.id,
        "text": rec.text,
        "tokens": tokens,
        "tags": token_labels
    })
print(f"Примеры собраны: {len(examples)}")

# Посмотрим пример
print("Пример tokens/tags:", examples[2]["tokens"][:20], examples[2]["tags"][:20])

Примеры собраны: 254
Пример tokens/tags: ['30', 'июля', 'в', 'Москве', 'состоялось', 'заседание', 'российского', 'правительства,', 'на', 'котором', 'рассматривался', 'вопрос', 'о', 'ценообразовании', 'на', 'лекарства,', 'как', 'отечественного', 'производства,', 'так'] ['O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [5]:
from datasets import Dataset, DatasetDict

unique_labels = set()
for ex in examples:
    unique_labels.update(ex["tags"])
unique_labels.add("O")
label_list = sorted(unique_labels)
label2id = {lab: i for i, lab in enumerate(label_list)}
id2label = {i: lab for lab, i in label2id.items()}

for ex in examples:
    ex["tags"] = [label2id[t] for t in ex["tags"]]

full_ds = Dataset.from_list(examples)
split = full_ds.train_test_split(test_size=0.1, seed=42)
dataset = DatasetDict({"train": split["train"], "test": split["test"]})
print(dataset) 

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'tokens', 'tags'],
        num_rows: 228
    })
    test: Dataset({
        features: ['id', 'text', 'tokens', 'tags'],
        num_rows: 26
    })
})


In [6]:
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name, is_split_into_words=True)


def tokenize_and_align_labels(examples_batch):
    tokenized = tokenizer(
        examples_batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128
    )
    labels = []
    for i, word_labels in enumerate(examples_batch["tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != prev_word_idx:
                label_ids.append(word_labels[word_idx])
            else:
                label_ids.append(-100)
            prev_word_idx = word_idx
        labels.append(label_ids)
    tokenized["labels"] = labels
    return tokenized


# 8. Применяем токенизацию к датасету
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=["text", "tokens", "tags", "id"]
)

print(tokenized_dataset)
# Проверка: показываем один пример из train
print(tokenized_dataset["train"][0])

Map:   0%|          | 0/228 [00:00<?, ? examples/s]

Map:   0%|          | 0/26 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 228
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 26
    })
})
{'input_ids': [2, 105, 78457, 60905, 117, 37239, 16345, 4711, 1063, 54125, 761, 22018, 41477, 36386, 47401, 43190, 13863, 79295, 314, 32993, 18, 105, 78457, 60905, 117, 39480, 72891, 34425, 13863, 2595, 28530, 16345, 4711, 1063, 16, 4146, 42085, 16234, 17, 10709, 105, 15, 793, 74659, 117, 16, 22502, 105, 45181, 117, 18, 47799, 75067, 23004, 41230, 105, 78457, 117, 314, 1683, 4412, 16908, 18, 299, 41869, 7197, 18200, 69704, 320, 42527, 31222, 51831, 1841, 30416, 21646, 9790, 2798, 1241, 34425, 32719, 18, 26129, 16345, 4711, 1063, 6744, 32606, 314, 28940, 30494, 13815, 4219, 2313, 26403, 105, 78457, 60905, 117, 18, 105, 12810, 27952, 13409, 117, 36170, 314, 5256, 3452, 616, 735, 18, 5971, 36236, 16746, 4596, 6543, 58082

In [7]:
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader

data_collator = DataCollatorForTokenClassification(tokenizer)

train_dataloader = DataLoader(
    tokenized_dataset["train"],
    batch_size=16,
    shuffle=True,
    collate_fn=data_collator
)
print("Готово. Примеры для обучения:", len(tokenized_dataset["train"]))

Готово. Примеры для обучения: 228


# Train

In [8]:
# Функция, выравнивающая предсказания модели и реальные метки (на уровне tokenized_dataset)
def get_flat_labels_and_preds_from_model(tokenized_split, model, device, max_samples=None):
    """
    tokenized_split: dataset split (list-like of examples with keys 'input_ids','attention_mask','labels')
    Возвращает flat lists: y_true (ints), y_pred (ints)
    """
    y_true = []
    y_pred = []
    for i, ex in enumerate(tokenized_split):
        if max_samples is not None and i >= max_samples:
            break

        # Превращаем в тензоры (batch size = 1)
        input_ids = torch.tensor([ex["input_ids"]], dtype=torch.long).to(device)
        attention_mask = torch.tensor([ex["attention_mask"]], dtype=torch.long).to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits  # shape (1, seq_len, num_labels)
            preds = torch.argmax(logits, dim=-1).squeeze(0).cpu().tolist()  # list длины seq_len

        # Истинные метки (включая -100 для пэддинга/ignored)
        true_labels = ex["labels"]  # список длиной seq_len; элементы -100 или id

        # Фильтруем позиции, где true != -100
        filtered_true = []
        filtered_pred = []
        for p, t in zip(preds, true_labels):
            if t == -100:
                continue
            filtered_true.append(int(t))
            filtered_pred.append(int(p))

        # Обрежем на минимальную длину (на случай рассинхронизации)
        minlen = min(len(filtered_true), len(filtered_pred))
        if minlen == 0:
            continue
        y_true.extend(filtered_true[:minlen])
        y_pred.extend(filtered_pred[:minlen])

    return y_true, y_pred

In [9]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# После этой строки обязательно перезапустите Kernel (Kernel → Restart)

In [11]:
import torch
from transformers import AutoModelForTokenClassification
from sklearn.metrics import precision_score, recall_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Загружаем модель 
model = AutoModelForTokenClassification.from_pretrained(
    "cointegrated/rubert-tiny2",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
    
)

model.to(device)
model.eval()

y_true, y_pred = get_flat_labels_and_preds_from_model(
    tokenized_dataset["test"], model, device, max_samples=200
)

print("Samples used (token-level):", len(y_true))
print("Precision:", precision_score(y_true, y_pred, average="macro", zero_division=0))
print("Recall:   ", recall_score (y_true, y_pred, average="macro", zero_division=0))
print("F1:       ", f1_score   (y_true, y_pred, average="macro", zero_division=0))

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expec

Samples used (token-level): 2315
Precision: 0.10781024566405804
Recall:    0.07326325072258034
F1:        0.06533722784694582


In [12]:
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader
import torch
from tqdm import tqdm

num_epochs = 5
learning_rate = 5e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    n_batches = 0
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}"):
        # Переносим тензоры на device
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        n_batches += 1
    avg_loss = total_loss / n_batches if n_batches > 0 else 0.0
    print(f"Epoch {epoch+1} avg loss: {avg_loss:.4f}")

# Переводим модель в eval перед оценкой
model.eval()
y_true_ft, y_pred_ft = get_flat_labels_and_preds_from_model(
    tokenized_dataset["test"], model, device, max_samples=200
)

print("After fine-tuning:")
print("Precision:", precision_score(y_true_ft, y_pred_ft, average="macro", zero_division=0))
print("Recall:   ", recall_score (y_true_ft, y_pred_ft, average="macro", zero_division=0))
print("F1:       ", f1_score   (y_true_ft, y_pred_ft, average="macro", zero_division=0))

print(f"Delta F1: {f1_score(y_true_ft, y_pred_ft, average='macro') - f1_score(y_true, y_pred, average='macro'):.4f}")

Epoch 1: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [00:00<00:00, 38.00it/s]


Epoch 1 avg loss: 1.2649


Epoch 2: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [00:00<00:00, 63.24it/s]


Epoch 2 avg loss: 0.7261


Epoch 3: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [00:00<00:00, 61.92it/s]


Epoch 3 avg loss: 0.5998


Epoch 4: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [00:00<00:00, 67.60it/s]


Epoch 4 avg loss: 0.4862


Epoch 5: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [00:00<00:00, 65.88it/s]

Epoch 5 avg loss: 0.3891
After fine-tuning:
Precision: 0.3519091036790152
Recall:    0.41108712501604733
F1:        0.37820283167497964
Delta F1: 0.3129
